# Limpieza, análisis y estructuración de datos de Instacart con Pandas

**Actividad:** GA1-220501094-01-AA1-EV06 – Limpieza, análisis y estructuración de datos con Pandas (Instacart)
**Aprendiz:** Liliana Sofia Panesso Mendez

**Pregunta orientadora:** ¿Cómo preparar y estructurar los datos de Instacart para obtener información confiable sobre los hábitos de compra?

Este notebook desarrolla, en orden, las lecciones 2, 3 y 4 del material *Preparar y estructurar datos con Python*: ejemplos, laboratorio guiado y reto de cada una. Cada lección parte de las tablas que deja la anterior, por eso el notebook debe ejecutarse completo y de arriba hacia abajo (*Kernel → Restart Kernel and Run All Cells*).

> Las rutas del material (`datos/crudos`, `datos/procesados`) se adaptaron a la estructura del proyecto en inglés: `data/raw` y `data/processed`.

## Preparación

In [1]:
# P.1 Librerías y rutas
import pandas as pd
import numpy as np
from pathlib import Path

RUTA = Path('../data/raw')
SALIDA = Path('../data/processed')

print(pd.__version__)

3.1.0


Se importaron las librerías y se definieron las rutas de entrada (`data/raw`) y salida (`data/processed`). La versión de pandas es **3.1.0**.

In [2]:
# P.2 Carga de las cinco tablas
orders      = pd.read_csv(RUTA / 'instacart_orders.csv', sep=';')
order_prods = pd.read_csv(RUTA / 'order_products.csv', sep=';')
products    = pd.read_csv(RUTA / 'products.csv', sep=';')
aisles      = pd.read_csv(RUTA / 'aisles.csv', sep=';')
departments = pd.read_csv(RUTA / 'departments.csv', sep=';')

Se cargaron las cinco tablas con `sep=';'`, porque los archivos usan punto y coma como separador.

In [3]:
# P.3 Verificación de tamaños (Lección 1)
for nombre, df in [('orders', orders), ('order_prods', order_prods), ('products', products),
                   ('aisles', aisles), ('departments', departments)]:
    print(nombre, df.shape)

orders (478967, 6)
order_prods (4545007, 4)
products (49694, 4)
aisles (134, 2)
departments (21, 2)


Los tamaños **coinciden** con los esperados en la Lección 1: `orders (478967, 6)`, `order_prods (4545007, 4)`, `products (49694, 4)`, `aisles (134, 2)` y `departments (21, 2)`.

## Lección 2: Limpieza y estandarización de datos crudos

Se resuelven los problemas encontrados en la exploración (actividad EV04 y Lección 1): duplicados, tipos de dato, nulos, texto sin estandarizar y duplicados implícitos. Para cada problema se decide entre **eliminar, corregir o conservar**, siguiendo este orden: duplicados, tipos, nulos, texto, duplicados implícitos y verificación.

### Ejemplos de la lección

#### Duplicados exactos

In [4]:
# L2.E1 Ver los duplicados antes de borrarlos
duplicados = orders[orders.duplicated(keep=False)].sort_values('order_id')
print(len(duplicados))
duplicados[['order_id', 'order_dow', 'order_hour_of_day']].head(6)

30


,order_id,order_dow,order_hour_of_day
354993,391768,3,2
371905,391768,3,2
119251,408114,3,2
321100,408114,3,2
394347,467134,3,2
250626,467134,3,2


Se muestran **30** filas: 15 pedidos y sus 15 copias, como se esperaba. Todas tienen `order_dow = 3` y `order_hour_of_day = 2`: que ocurran en la misma franja apunta a un fallo del sistema y no a pedidos reales, así que es seguro borrarlas.

In [5]:
# L2.E2 Eliminar los duplicados
orders = orders.drop_duplicates().reset_index(drop=True)
print(orders.shape, orders['order_id'].is_unique)

(478952, 6) True


Tras `drop_duplicates()` el resultado es **`(478952, 6) True`**, igual al esperado: quedan 478 952 pedidos y `order_id` ya es único.

#### Tipos de dato con `astype`

In [6]:
# L2.E3 Corregir tipos y reducir memoria
print('Antes:', round(orders.memory_usage(deep=True).sum() / 1e6, 1), 'MB')

orders = orders.astype({
    'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
    'order_dow': 'int8', 'order_hour_of_day': 'int8',
    'days_since_prior_order': 'Int16',
})
order_prods = order_prods.astype({
    'order_id': 'int32', 'product_id': 'int32',
    'add_to_cart_order': 'Int16', 'reordered': 'int8',
})

print('Después:', round(orders.memory_usage(deep=True).sum() / 1e6, 1), 'MB')

Antes: 23.0 MB


Después: 7.2 MB


La memoria de `orders` bajó de **23,0 MB** a **7,2 MB**, como indica el material. Se usó `Int16` (con mayúscula) en las columnas con nulos, porque admite enteros y nulos a la vez.

#### Nulos: comprobar la causa antes de decidir

In [7]:
# L2.E4 ¿Los nulos de days_since_prior_order son primeros pedidos?
nulos = orders['days_since_prior_order'].isna()
primeros = orders['order_number'] == 1
print((nulos == primeros).all())

True


El resultado es **`True`**: cada nulo de `days_since_prior_order` es un primer pedido. Decisión: **conservar** el nulo; rellenarlo con 0 inventaría que el cliente volvió el mismo día.

In [8]:
# L2.E5 Productos sin nombre
print(products[products['product_name'].isna()][['aisle_id', 'department_id']].value_counts())
products['product_name'] = products['product_name'].fillna('Unknown')
print(products['product_name'].isna().sum())

aisle_id  department_id
100       21               1258
Name: count, dtype: int64
0


Hay una sola combinación, pasillo **100** y departamento **21** (`missing`), con **1 258** productos; después de `fillna` quedan **0** nulos. Coincide con lo esperado. Decisión: **corregir** con `'Unknown'`, porque el producto existe y se usa en pedidos.

| Columna | Causa | Decisión |
| --- | --- | --- |
| `days_since_prior_order` | Primer pedido del cliente | Conservar el nulo |
| `product_name` | Producto sin datos de catálogo | Rellenar con `'Unknown'` |
| `add_to_cart_order` | Se investiga en el Reto 2 (ejercicio 3) | Conservar el nulo |

#### Texto con los métodos `.str`

In [9]:
# L2.E6 Estandarizar los nombres de producto
nombres = products['product_name']
print('Espacios dobles:', nombres.str.contains('  ', regex=False).sum())
print('Espacio invisible:', nombres.str.contains('\xa0', regex=False).sum())

products['product_name'] = (
    products['product_name']
    .str.replace('\xa0', ' ', regex=False)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
    .str.title()
)
print(products['product_name'].str.contains('  ', regex=False).sum(),
      products['product_name'].str.contains('\xa0', regex=False).sum())

Espacios dobles: 403
Espacio invisible: 16


0 0


Había **403** nombres con espacios dobles y **16** con espacio invisible (`\xa0`); después de limpiar quedan **0 0**. Coincide con el material.

#### Duplicados implícitos

In [10]:
# L2.E7 Nombres repetidos después de estandarizar
conocidos = products[products['product_name'] != 'Unknown']
repetidos = conocidos[conocidos['product_name'].duplicated(keep=False)]
print(conocidos['product_name'].duplicated().sum())
print(repetidos['product_id'].isin(order_prods['product_id']).sum(), 'de', len(repetidos))

141


268 de 280


Aparecen **141** nombres repetidos tras estandarizar y **268 de 280** productos implicados aparecen en pedidos, igual que en el material. Decisión: **conservar y documentar**, porque tienen `product_id` distintos; en los análisis se agrupa por `product_id`.

#### Limpieza reutilizable con funciones

In [11]:
# L2.E8 Funciones de limpieza
def limpiar_orders(df):
    df = df.drop_duplicates().reset_index(drop=True)
    df = df.astype({'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
                    'order_dow': 'int8', 'order_hour_of_day': 'int8',
                    'days_since_prior_order': 'Int16'})
    assert df['order_id'].is_unique
    return df

def limpiar_products(df):
    df = df.copy()
    df['product_name'] = (df['product_name'].fillna('Unknown')
                          .str.replace('\xa0', ' ', regex=False)
                          .str.replace(r'\s+', ' ', regex=True)
                          .str.strip().str.title())
    return df

print(limpiar_orders(orders).shape)

(478952, 6)


La función devuelve **`(478952, 6)`**: aplicada a una tabla ya limpia no cambia nada, como se esperaba.

In [12]:
# L2.E9 Verificar toda la limpieza
assert orders.shape == (478952, 6)
assert orders['days_since_prior_order'].isna().sum() == 28817   # 2 nulos estaban entre los duplicados
assert products['product_name'].isna().sum() == 0
assert str(order_prods['add_to_cart_order'].dtype) == 'Int16'
print('Verificación superada')

Verificación superada


Los cuatro `assert` pasaron sin error (se agregó un `print` para dejar constancia visible). Hay **28 817** nulos en `days_since_prior_order` porque 2 de los nulos originales estaban entre los duplicados.

### Laboratorio guiado 2: pipeline de limpieza con informe de cambios

Se construye un **pipeline**: una secuencia de funciones encadenadas que parte de los archivos crudos, se ejecuta con una sola instrucción y deja un informe de lo que cambió en cada paso. Se trabaja sobre copias nuevas, sin tocar las tablas ya limpias.

#### Paso 1: recargar los datos crudos

In [13]:
# L2.LAB1 Recargar los datos crudos
orders_crudo   = pd.read_csv(RUTA / 'instacart_orders.csv', sep=';')
products_crudo = pd.read_csv(RUTA / 'products.csv', sep=';')
op_crudo       = pd.read_csv(RUTA / 'order_products.csv', sep=';')
print(orders_crudo.shape, products_crudo.shape, op_crudo.shape)

(478967, 6) (49694, 4) (4545007, 4)


Se recargaron los archivos crudos: **`(478967, 6) (49694, 4) (4545007, 4)`**, los tamaños originales.

#### Paso 2: una función que anota el estado de la tabla

In [14]:
# L2.LAB2 Función de registro
registro = []

def anotar(df, tabla, paso):
    registro.append({
        'tabla': tabla,
        'paso': paso,
        'filas': len(df),
        'nulos': int(df.isna().sum().sum()),
        'memoria_mb': round(df.memory_usage(deep=True).sum() / 1e6, 1),
    })
    return df

`anotar` guarda una "foto" (filas, nulos, memoria) y devuelve la tabla **sin cambios**, para poder intercalarla entre pasos.

#### Paso 3: un paso de limpieza por función

In [15]:
# L2.LAB3 Pasos de limpieza
def quitar_duplicados(df):
    return df.drop_duplicates().reset_index(drop=True)

def tipos_orders(df):
    return df.astype({'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
                      'order_dow': 'int8', 'order_hour_of_day': 'int8',
                      'days_since_prior_order': 'Int16'})

def limpiar_nombres(df):
    df = df.copy()
    df['product_name'] = (df['product_name'].fillna('Unknown')
                          .str.replace('\xa0', ' ', regex=False)
                          .str.replace(r'\s+', ' ', regex=True)
                          .str.strip().str.title())
    return df

def tipos_order_prods(df):
    return df.astype({'order_id': 'int32', 'product_id': 'int32',
                      'add_to_cart_order': 'Int16', 'reordered': 'int8'})

Cada función hace un solo paso de limpieza y recibe y devuelve un DataFrame; esa regla permite encadenarlas.

#### Paso 4: encadenar los pasos con `pipe`

In [16]:
# L2.LAB4 Pipeline
orders_limpio = (orders_crudo
    .pipe(anotar, 'orders', '0 crudo')
    .pipe(quitar_duplicados)
    .pipe(anotar, 'orders', '1 sin duplicados')
    .pipe(tipos_orders)
    .pipe(anotar, 'orders', '2 tipos'))

products_limpio = (products_crudo
    .pipe(anotar, 'products', '0 crudo')
    .pipe(limpiar_nombres)
    .pipe(anotar, 'products', '1 nombres'))

op_limpio = (op_crudo
    .pipe(anotar, 'order_prods', '0 crudo')
    .pipe(tipos_order_prods)
    .pipe(anotar, 'order_prods', '1 tipos'))

El pipeline se lee de arriba abajo como una receta: crudo → quitar duplicados → anotar → corregir tipos → anotar. La celda se ejecutó una sola vez, por eso `registro` no tiene filas repetidas.

#### Paso 5: el informe de cambios

In [17]:
# L2.LAB5 Informe de cambios
informe = pd.DataFrame(registro)
informe

,tabla,paso,filas,nulos,memoria_mb
0,orders,0 crudo,478967,28819,23.0
1,orders,1 sin duplicados,478952,28817,23.0
2,orders,2 tipos,478952,28817,7.2
3,products,0 crudo,49694,1258,3.1
4,products,1 nombres,49694,0,3.1
5,order_prods,0 crudo,4545007,836,145.4
6,order_prods,1 tipos,4545007,836,54.5


El informe **coincide** con la tabla esperada del material:

- Quitar duplicados eliminó **15 filas** y **2 nulos** en `orders` (478 967 → 478 952; 28 819 → 28 817): dos de los pedidos repetidos eran primeros pedidos.
- Corregir tipos no cambió filas ni nulos, pero redujo la memoria de `orders` de **23,0 a 7,2 MB** y la de `order_prods` de **145,4 a 54,5 MB**.
- Los **1 258** nulos de `products` pasaron a 0 al rellenar con `'Unknown'`.
- Los **836** nulos de `order_prods` se conservan a propósito.

#### Paso 6: validar el resultado

In [18]:
# L2.LAB6 Validación
def validar(o, p, op):
    assert o['order_id'].is_unique, 'order_id repetido'
    assert o['order_hour_of_day'].between(0, 23).all(), 'hora fuera de rango'
    assert p['product_name'].notna().all(), 'productos sin nombre'
    assert not p['product_name'].str.contains('\xa0', regex=False).any(), 'espacios invisibles'
    assert op['reordered'].isin([0, 1]).all(), 'reordered con valores raros'
    return 'Validación superada'

print(validar(orders_limpio, products_limpio, op_limpio))
print(orders_limpio.equals(orders), products_limpio.equals(products))

Validación superada
True True


Resultado: **`Validación superada`** y **`True True`**, como se esperaba: el pipeline produce exactamente las mismas tablas que la limpieza manual de los ejemplos.

### Reto 2

#### Ejercicio 1
Comprueba si `order_prods` tiene filas duplicadas exactas, si se repite algún par `order_id` + `product_id` y si `reordered` solo contiene 0 y 1.

In [19]:
# L2.R1 Duplicados y valores de reordered en order_prods
print('Filas duplicadas exactas:', order_prods.duplicated().sum())
print('Pares order_id + product_id repetidos:', order_prods.duplicated(subset=['order_id', 'product_id']).sum())
print('Valores de reordered:', sorted(order_prods['reordered'].unique().tolist()))
print('¿Solo 0 y 1?:', order_prods['reordered'].isin([0, 1]).all())

Filas duplicadas exactas: 0


Pares order_id + product_id repetidos: 0
Valores de reordered: [0, 1]
¿Solo 0 y 1?: True


**Respuesta:** `order_prods` no tiene filas duplicadas exactas (**0**) ni pares `order_id` + `product_id` repetidos (**0**), y `reordered` solo contiene **0 y 1**. No necesita limpieza de duplicados.

#### Ejercicio 2
Comprueba si `aisles` y `departments` tienen nombres duplicados o con espacios al inicio o al final. ¿Necesitan limpieza de texto?

In [20]:
# L2.R2 Nombres de pasillos y departamentos
for nombre, df, col in [('aisles', aisles, 'aisle'), ('departments', departments, 'department')]:
    texto = df[col]
    print(nombre)
    print('  nombres duplicados:', texto.duplicated().sum())
    print('  con espacios al inicio o al final:', (texto != texto.str.strip()).sum())
    print('  con espacios dobles:', texto.str.contains('  ', regex=False).sum())
    print('  con espacio invisible:', texto.str.contains('\xa0', regex=False).sum())

aisles
  nombres duplicados: 0
  con espacios al inicio o al final: 0
  con espacios dobles: 0
  con espacio invisible: 0
departments
  nombres duplicados: 0
  con espacios al inicio o al final: 0
  con espacios dobles: 0
  con espacio invisible: 0


**Respuesta:** ni `aisles` ni `departments` tienen nombres duplicados, espacios al inicio o al final, espacios dobles ni espacios invisibles (**0** en todos los casos). No necesitan limpieza de texto.

#### Ejercicio 3
Investiga los 836 nulos de `add_to_cart_order`: ¿en cuántos pedidos ocurren y cuántos productos tiene, como mínimo, cada uno? ¿Qué los explica?

In [21]:
# L2.R3 Pedidos con nulos en add_to_cart_order
nulos_carrito = order_prods['add_to_cart_order'].isna()
pedidos_con_nulos = order_prods.loc[nulos_carrito, 'order_id'].unique()
print('Nulos:', nulos_carrito.sum())
print('Pedidos con nulos:', len(pedidos_con_nulos))

tamano = order_prods[order_prods['order_id'].isin(pedidos_con_nulos)].groupby('order_id').size()
print('Mínimo de productos en esos pedidos:', tamano.min())
print('Máximo de add_to_cart_order registrado en toda la tabla:', order_prods['add_to_cart_order'].max())
print('Pedidos con más de 64 productos en toda la tabla:', (order_prods.groupby('order_id').size() > 64).sum())

Nulos: 836
Pedidos con nulos: 70
Mínimo de productos en esos pedidos: 65
Máximo de add_to_cart_order registrado en toda la tabla: 64


Pedidos con más de 64 productos en toda la tabla: 70


**Respuesta:** los **836** nulos están en **70** pedidos, y cada uno tiene como mínimo **65** productos. El valor máximo registrado de `add_to_cart_order` es **64** y hay exactamente **70** pedidos con más de 64 productos: el sistema solo registró la posición hasta la 64, y los productos posteriores quedaron sin posición. Por eso se conservan como nulos.

#### Ejercicio 4
Carga de nuevo `products.csv` en una variable `products_crudo` y cuenta cuántos nombres (no nulos) cambiaron con la estandarización.

In [22]:
# L2.R4 Nombres que cambiaron con la estandarización
products_crudo = pd.read_csv(RUTA / 'products.csv', sep=';')
tenia_nombre = products_crudo['product_name'].notna()
cambiaron = products_crudo.loc[tenia_nombre, 'product_name'] != products.loc[tenia_nombre, 'product_name']
print('Nombres no nulos:', tenia_nombre.sum())
print('Nombres que cambiaron:', cambiaron.sum())
print(f'Porcentaje: {cambiaron.mean() * 100:.2f} %')

Nombres no nulos: 48436
Nombres que cambiaron: 6321
Porcentaje: 13.05 %


**Respuesta:** de los **48 436** nombres no nulos, **6 321** (**13,05 %**) cambiaron con la estandarización (espacios y mayúsculas iniciales).

#### Ejercicio 5
Crea una Series con el nombre del día de cada pedido y conviértela a una categoría **ordenada** de domingo a sábado. Compara la memoria del texto con la de la categoría.

In [23]:
# L2.R5 Día de la semana como categoría ordenada
dias = {0: 'domingo', 1: 'lunes', 2: 'martes', 3: 'miércoles',
        4: 'jueves', 5: 'viernes', 6: 'sábado'}
orden_dias = ['domingo', 'lunes', 'martes', 'miércoles', 'jueves', 'viernes', 'sábado']

dia_texto = orders['order_dow'].map(dias)
dia_categoria = pd.Series(pd.Categorical(dia_texto, categories=orden_dias, ordered=True))

print('Tipo texto:', dia_texto.dtype, '| memoria:', round(dia_texto.memory_usage(deep=True) / 1e6, 2), 'MB')
print('Tipo categoría:', dia_categoria.dtype, '| memoria:', round(dia_categoria.memory_usage(deep=True) / 1e6, 2), 'MB')
print('¿Ordenada?:', dia_categoria.cat.ordered)
print('Primera y última categoría:', dia_categoria.min(), '-', dia_categoria.max())

Tipo texto: str | memoria: 7.14 MB
Tipo categoría: category | memoria: 0.48 MB
¿Ordenada?: True
Primera y última categoría: domingo - sábado


**Respuesta:** la Series de texto ocupa **7,14 MB** y la categoría ordenada **0,48 MB**, unas 15 veces menos, porque guarda cada nombre de día una sola vez. Al estar ordenada, el mínimo es *domingo* y el máximo *sábado*.

#### Ejercicio 6
¿Cuántas filas de `order_prods` corresponden a productos `'Unknown'` y en cuántos pedidos aparecen? ¿Los eliminarías? Justifica.

In [24]:
# L2.R6 Productos 'Unknown' en los pedidos
ids_unknown = products.loc[products['product_name'] == 'Unknown', 'product_id']
filas_unknown = order_prods[order_prods['product_id'].isin(ids_unknown)]
print('Productos Unknown en el catálogo:', len(ids_unknown))
print('Filas de order_prods con productos Unknown:', len(filas_unknown))
print(f'Porcentaje de filas: {len(filas_unknown) / len(order_prods) * 100:.2f} %')
print('Pedidos en los que aparecen:', filas_unknown['order_id'].nunique())

Productos Unknown en el catálogo: 1258
Filas de order_prods con productos Unknown: 9547
Porcentaje de filas: 0.21 %
Pedidos en los que aparecen: 8254


**Respuesta:** hay **9 547** filas de `order_prods` (**0,21 %**) con productos `'Unknown'`, repartidas en **8 254** pedidos. **No los eliminaría**: son compras reales; borrarlas quitaría productos de 8 254 pedidos y alteraría conteos como el tamaño de los pedidos o la tasa de recompra. Basta con identificarlos.

#### Ejercicio 7
En una **copia** de `departments`, reemplaza `'missing'` por un nulo real (`pd.NA`). Explica en una frase por qué no conviene hacerlo en la tabla original antes de agrupar.

In [25]:
# L2.R7 'missing' como nulo real en una copia
departments_na = departments.copy()
departments_na['department'] = departments_na['department'].replace('missing', pd.NA)
print('Nulos en la copia:', departments_na['department'].isna().sum())
print('Nulos en la original:', departments['department'].isna().sum())
departments_na.tail(3)

Nulos en la copia: 1
Nulos en la original: 0


,department_id,department
18,19,snacks
19,20,deli
20,21,NaN


En la copia, `'missing'` pasó a ser un nulo real (**1** nulo), mientras que la tabla original sigue con **0** nulos.

In [26]:
# L2.R7b Efecto de un nulo al agrupar
prueba = products.merge(departments_na, on='department_id', how='left')
print('Productos por departamento (original con missing):',
      products.merge(departments, on='department_id').groupby('department').size().sum())
print('Productos por departamento (copia con NA):', prueba.groupby('department').size().sum())
print('Productos que desaparecen del groupby:', prueba['department'].isna().sum())

Productos por departamento (original con missing): 49694
Productos por departamento (copia con NA): 48436
Productos que desaparecen del groupby: 1258


**Respuesta:** no conviene hacerlo en la tabla original porque `groupby` descarta los nulos por defecto: en esta prueba, **1 258** productos desaparecen del agrupamiento (49 694 frente a 48 436), y los totales dejarían de cuadrar con el número real de productos.

### Conclusiones de la Lección 2

- **Duplicados:** se eliminaron 15 pedidos repetidos, todos del día 3 a las 2 h. `orders` quedó con 478 952 filas y `order_id` único.
- **Tipos:** con `astype` la memoria bajó de 23,0 a 7,2 MB en `orders` y de 145,4 a 54,5 MB en `order_prods`. Los tipos `Int16` permiten conservar los nulos.
- **Nulos:** cada nulo se decidió según su causa: se conservaron los 28 817 de primeros pedidos y los 836 de posiciones mayores a 64 en el carrito, y se rellenaron con `'Unknown'` los 1 258 nombres faltantes.
- **Texto:** la estandarización cambió 6 321 nombres y dejó 141 duplicados implícitos, que se conservan porque tienen `product_id` distintos.
- **Pipeline:** el Laboratorio 2 reproduce la limpieza manual con funciones encadenadas (`True True`) y deja un informe de cambios con cifras.